# W06 — Advanced ensembles and metric-aligned ranking

**Question:** Can stacking, ensemble selection, or a ranking-objective change improve the current 92.4% mean P@50 by at least 5 percentage points on the same five client-grouped folds?

The experiment follows four primary sources: [Wolpert's stacked generalization](https://doi.org/10.1016/S0893-6080(05)80023-1) and [Super Learner](https://pubmed.ncbi.nlm.nih.gov/17910531/) (level-2 learning from held-out predictions), [Caruana et al.'s ensemble selection](https://www.cs.cornell.edu/~caruana/ctp/ct.papers/caruana.icml04.icdm06long.pdf) (diverse libraries optimized against the operational metric), and [Burges' LambdaMART overview](https://www.microsoft.com/en-us/research/wp-content/uploads/2016/02/MSR-TR-2010-82.pdf) (ranking losses shaped by top-of-list utility).

This remains a **current-snapshot cross-client ranking** experiment, not future forecasting. Target-window fields, label-derived fields, identifiers, and product scores are excluded from all model feature matrices.

In [1]:
import json
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
OUTPUTS = ROOT / 'work' / 'outputs'
advanced = json.loads((OUTPUTS / 'advanced_ranking_results.json').read_text(encoding='utf-8'))
stacking = json.loads((OUTPUTS / 'stacking_probe_results.json').read_text(encoding='utf-8'))
blends = json.loads((OUTPUTS / 'advanced_blend_search_results.json').read_text(encoding='utf-8'))
hierarchy = json.loads((OUTPUTS / 'hierarchical_calibration_probe_results.json').read_text(encoding='utf-8'))

print(f"Rows: {advanced['rows']:,}; clients: {advanced['clients']}; folds: 5")
print(f"Estimand: {advanced['estimand']}")
print(f"Primary metric: {advanced['primary_metric']}")
print('No row identifiers, client names, URLs, or queries are displayed.')

Rows: 30,000; clients: 32; folds: 5
Estimand: cross-client current-snapshot decline ranking; not future forecasting
Primary metric: mean precision@50 across the same five fixed GroupKFold folds
No row identifiers, client names, URLs, or queries are displayed.


## Experimental design

1. Freeze the five `GroupKFold(client_id)` outer folds used by W6 and FE v1/v2.
2. Screen conventional stacking with fold-ranked OOF scores, regularized linear/tree meta-learners, and a top-candidate boundary reranker.
3. Test LambdaRank query definitions that mirror the pooled multi-client queue, plus `rank_xendcg` and top-20 truncation.
4. Test fixed rank blends, cross-fitted blend-weight selection, and hierarchical client calibration.
5. Treat fold-wise oracle selection as a ceiling only; it is never a valid reported model.

In [2]:
comparison = pd.DataFrame(advanced['comparison'])
display_columns = [
    'candidate', 'mean_p_at_20', 'mean_p_at_50', 'std_p_at_50',
    'mean_p_at_100', 'mean_roc_auc', 'mean_average_precision'
]
table = comparison[display_columns].head(8).copy()
for column in display_columns[1:]:
    table[column] = table[column].map(lambda value: f'{value:.1%}')
print(table.to_string(index=False))
print(f"\nCurrent benchmark P@50: {advanced['benchmark_mean_p_at_50']:.1%}")
print(f"Best advanced candidate: {advanced['winner']} at {advanced['winner_mean_p_at_50']:.1%}")
print(f"Observed improvement: {advanced['improvement_pp']:+.1f}pp")
print(f"Requested target: +{advanced['target_improvement_pp']:.1f}pp; achieved: {advanced['target_achieved']}")

                       candidate mean_p_at_20 mean_p_at_50 std_p_at_50 mean_p_at_100 mean_roc_auc mean_average_precision
    A11_current_plus_top20_equal        94.0%        92.8%        2.3%         88.4%        64.2%                  68.3%
           B2_current_rank_blend        92.0%        92.4%        3.3%         88.6%        64.3%                  68.4%
   A10_current_plus_xendcg_equal        92.0%        92.4%        3.3%         88.6%        64.3%                  68.4%
        B0_engineered_lambdarank        94.0%        91.6%        4.6%         85.0%        62.8%                  67.0%
           A7_three_ranker_equal        93.0%        91.2%        1.1%         87.8%        65.0%                  68.7%
              B1_base_lambdarank        90.0%        90.8%        2.7%         90.4%        64.7%                  68.5%
A6_current_plus_fold_query_equal        90.0%        90.8%        2.3%         88.0%        64.6%                  68.3%
   A3_base_fold_query_lambdarank

In [3]:
winner_folds = pd.DataFrame(advanced['results'][advanced['winner']]['folds'])
benchmark_folds = pd.DataFrame(advanced['results'][advanced['benchmark']]['folds'])
paired = pd.DataFrame({
    'fold': winner_folds['fold'],
    'benchmark_p50': benchmark_folds['p_at_50'],
    'advanced_p50': winner_folds['p_at_50'],
})
paired['change_pp'] = 100 * (paired['advanced_p50'] - paired['benchmark_p50'])
shown = paired.copy()
shown['benchmark_p50'] = shown['benchmark_p50'].map(lambda value: f'{value:.1%}')
shown['advanced_p50'] = shown['advanced_p50'].map(lambda value: f'{value:.1%}')
shown['change_pp'] = shown['change_pp'].map(lambda value: f'{value:+.1f}')
print(shown.to_string(index=False))
print(f"\nFolds improved: {(paired['change_pp'] > 0).sum()} / 5")
print(f"P@20 changed from {comparison.loc[comparison.candidate.eq(advanced['benchmark']), 'mean_p_at_20'].iloc[0]:.1%} to {comparison.loc[comparison.candidate.eq(advanced['winner']), 'mean_p_at_20'].iloc[0]:.1%}.")

 fold benchmark_p50 advanced_p50 change_pp
    1         88.0%        96.0%      +8.0
    2         94.0%        90.0%      -4.0
    3         94.0%        94.0%      +0.0
    4         90.0%        92.0%      +2.0
    5         96.0%        92.0%      -4.0

Folds improved: 2 / 5
P@20 changed from 92.0% to 94.0%.


In [4]:
diagnostics = pd.DataFrame([
    {
        'method': 'Best cross-fitted score stack',
        'p_at_50': stacking['winner_mean_p_at_50'],
        'status': 'rejected: below benchmark',
    },
    {
        'method': 'Cross-fitted blend-weight selection',
        'p_at_50': blends['cross_fitted_mean_p_at_50'],
        'status': 'no gain',
    },
    {
        'method': 'Hierarchical client calibration',
        'p_at_50': hierarchy['calibrated_mean_p_at_50'],
        'status': 'selected zero adjustment',
    },
    {
        'method': 'Fold-wise oracle blend',
        'p_at_50': blends['oracle_foldwise_mean_p_at_50_not_valid'],
        'status': 'INVALID: uses scored-fold labels',
    },
])
diagnostics['p_at_50'] = diagnostics['p_at_50'].map(lambda value: f'{value:.1%}')
print(diagnostics.to_string(index=False))
print('\nInterpretation: the library contains enough complementary predictions to reach the target only with fold-specific hindsight. The relationship does not transfer from the other folds.')

                             method p_at_50                           status
      Best cross-fitted score stack   89.6%        rejected: below benchmark
Cross-fitted blend-weight selection   92.4%                          no gain
    Hierarchical client calibration   92.4%         selected zero adjustment
             Fold-wise oracle blend   97.6% INVALID: uses scored-fold labels

Interpretation: the library contains enough complementary predictions to reach the target only with fold-specific hindsight. The relationship does not transfer from the other folds.


In [5]:
assert advanced['benchmark'] == 'B2_current_rank_blend'
assert abs(advanced['benchmark_mean_p_at_50'] - 0.924) < 1e-12
assert advanced['winner_mean_p_at_50'] >= advanced['benchmark_mean_p_at_50']
assert advanced['target_achieved'] is False
assert stacking['winner_mean_p_at_50'] < advanced['benchmark_mean_p_at_50']
assert abs(blends['cross_fitted_mean_p_at_50'] - advanced['benchmark_mean_p_at_50']) < 1e-12
assert abs(hierarchy['calibrated_mean_p_at_50'] - advanced['benchmark_mean_p_at_50']) < 1e-12
print('Validation receipt checks: PASS')
print('The +5pp claim is not supported by the honest experiments.')

Validation receipt checks: PASS
The +5pp claim is not supported by the honest experiments.


## Finding

The strongest honest treatment is an equal fold-rank blend of the current two-LambdaRank ensemble and a deliberately different top-20-truncated, fold-query LambdaRank. It reaches **92.8% ± 2.3% P@50**, versus **92.4% ± 3.3%** for the current model: **+0.4 percentage points**, while P@20 improves from 92% to 94%.

The requested +5pp target is **not achieved**. Conventional stacking (best 89.6%), boundary reranking (88.8%), cross-fitted adaptive weights (92.4%), and hierarchical calibration (92.4%) do not transfer. A 97.6% fold-wise oracle exists but is invalid because it chooses weights using each scored fold's labels.

**Recommendation:** retain the current 92.4% blend as the conservative benchmark; treat the 92.8% blend as a development candidate, not a confirmed production replacement. The next credible route to a material gain is new daily warehouse history with pre-cutoff slopes, volatility, seasonality, and multiple forecast origins—not more selection against this single snapshot.